In [11]:
import os
import pandas as pd
from IPython.display import FileLink, display

backup_dir = "/kaggle/working/HAR_Research"
os.makedirs(backup_dir, exist_ok=True)

manifest_path = os.path.join(
    backup_dir,
    "ucf101_split01_manifest.csv"
)

df.to_csv(manifest_path, index=False)

print("Manifest backup created.")
print("Total videos:", len(df))
print(df["split"].value_counts())

display(FileLink(manifest_path))

Manifest backup created.
Total videos: 13320
split
train         7933
test          3783
validation    1604
Name: count, dtype: int64


/kaggle/working/HAR_Research/ucf101_split01_manifest.csv

In [35]:

import time
import torch
from tqdm.auto import tqdm

def train_one_epoch(loader):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    start = time.time()

    progress = tqdm(loader, desc="Training Epoch 1")

    for videos, labels in progress:

        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Uniform selection: 4 frames from 29 candidates
        videos = uniform_sampling(videos, budget=4)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * correct / total:.2f}%"
        )

    return {
        "train_loss": total_loss / total,
        "train_accuracy": 100 * correct / total,
        "training_seconds": time.time() - start
    }


# Train epoch 1
results = train_one_epoch(train_loader)

print("\nTRAINING COMPLETED")
print("Training loss:", round(results["train_loss"], 4))
print("Training accuracy:", round(results["train_accuracy"], 2), "%")
print("Training time:", round(results["training_seconds"], 2), "seconds")

# Save checkpoint immediately
save_checkpoint(
    model,
    optimizer,
    epoch=1,
    filename="mobilenetv3_uniform_4frames_epoch1.pth"
)

print("\nFIRST BASELINE CHECKPOINT SAVED.")



Training Epoch 1:   0%|          | 0/3967 [00:00<?, ?it/s]


TRAINING COMPLETED
Training loss: 2.8082
Training accuracy: 33.48 %
Training time: 936.15 seconds
Checkpoint saved: /kaggle/working/HAR_Research/checkpoints/mobilenetv3_uniform_4frames_epoch1.pth
Size (MB): 49.92

FIRST BASELINE CHECKPOINT SAVED.


In [36]:

import os
import zipfile
import hashlib

checkpoint_path = (
    "/kaggle/working/HAR_Research/checkpoints/"
    "mobilenetv3_uniform_4frames_epoch1.pth"
)

assert os.path.exists(checkpoint_path), "Checkpoint missing!"

size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)

# Calculate SHA256 checksum for backup verification
sha256 = hashlib.sha256()

with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("CHECKPOINT VERIFIED")
print("File:", os.path.basename(checkpoint_path))
print("Size:", round(size_mb, 2), "MB")
print("SHA256:", sha256.hexdigest())

# Create a ZIP backup
zip_path = (
    "/kaggle/working/"
    "mobilenetv3_uniform_4frames_epoch1_backup.zip"
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    z.write(
        checkpoint_path,
        arcname=os.path.basename(checkpoint_path)
    )

print("\nZIP BACKUP CREATED")
print("Path:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")


CHECKPOINT VERIFIED
File: mobilenetv3_uniform_4frames_epoch1.pth
Size: 49.92 MB
SHA256: 36f0ebe17e88f6940cf081f63034c8571d25c1276b74556df4cfecbe9eeda6ea

ZIP BACKUP CREATED
Path: /kaggle/working/mobilenetv3_uniform_4frames_epoch1_backup.zip
Size: 49.92 MB


## Dataset Partitioning and Leakage Prevention

The UCF101 dataset was partitioned using the official Split 1 training and testing lists.

The official training data was further divided into training and validation subsets using a group-based stratified procedure within each action class.

- **Training:** 7,933 videos
- **Validation:** 1,604 videos
- **Testing:** 3,783 videos
- **Total:** 13,320 videos
- **Action classes:** 101

A fixed random seed (`random_state=42`) was used to ensure reproducibility. Source-group overlap between the training and validation subsets was checked and confirmed to be absent.

The official test set remains separate and will be reserved for final evaluation.

**Leakage Check:** PASSED.

### Stage 2: Dataset Integrity Audit — Completed

The complete UCF101 video integrity audit was successfully repeated.

- Total videos audited: 13,320
- Fully readable videos: 13,096
- Partially readable videos: 224
- Unreadable videos: 0
- Audit duration: 772.52 seconds

All videos contain at least one successfully decoded frame. Partially readable videos are retained using their actual decoded frame counts.

**Output:** `full_video_integrity_audit.csv`

**Next:** Restore the audited manifest, video Dataset class, and DataLoaders.

### Project Recovery — November 22, 2026

The Kaggle session restarted and the previous training checkpoint was not preserved.

**Previously completed work:**
- UCF101 dataset preparation and integrity audit.
- 32-frame candidate extraction pipeline.
- Uniform and temporal-difference sampling methods.
- MobileNetV3-Large video classifier.
- First baseline training epoch.

**Previously recorded baseline results:**
- Training videos: 7,933
- Frame budget: 4
- Training loss: 2.7532
- Training accuracy: 34.35%
- Training time: 901.24 seconds

**Recovery plan:** Restore the dataset pipeline, retrain the baseline, and immediately download the model checkpoint to local storage.

### Stage 1: Dataset Preparation and Model Pipeline Validation

**Completed tasks:**
- Verified the UCF101 dataset containing 13,320 videos across 101 action classes.
- Prepared training (7,933), validation (1,604), and testing (3,783) partitions.
- Completed video integrity auditing and generated an audited dataset manifest.
- Developed and tested sequential video decoding and 32-frame candidate extraction.
- Implemented and validated two frame-selection strategies:
  - Uniform temporal sampling
  - Temporal-difference-based adaptive sampling
- Tested frame budgets of 4, 8, 12, and 16.
- Configured MobileNetV3-Large with a 101-class output layer.
- Successfully completed a 10-batch GPU training pilot.

**Initial status:** The data pipeline, sampling methods, classification model, and initial GPU training workflow are operational.

**Next stage:** Conduct controlled baseline and proposed-method experiments, followed by validation and computational-efficiency evaluation.

*Note: The pilot is a pipeline test, not a final model-performance evaluation.*